In [ ]:
import torch

In [ ]:
# Cell 2
class SyntheticRegressionData:
    """정답을 알고 있는 선형회귀용 합성 데이터를 생성한다."""

    def __init__(
        self,
        weights,
        bias,
        noise_std=0.01,
        num_train=1000,
        num_val=1000,
        batch_size=32,
    ):
        # 데이터 생성에 사용한 정답과 설정값을 저장한다.
        self.weights = weights
        self.bias = bias 
        self.noise_std = noise_std
        self.num_train = num_train
        self.num_val = num_val
        self.batch_size = batch_size

        # 훈련 example과 validation example을 모두 생성한다.
        number_of_examples = num_train + num_val

        # 각 sample은 feature 2개를 가진다.
        number_of_features = weights.numel()

        # 표준정규분포 N(0, 1)에서 feature를 생성한다.
        #
        # X shape:
        # (number_of_examples, number_of_features)
        self.X = torch.randn(
            number_of_examples,
            number_of_features,
        )

        # 행렬곱을 위해 weights를 열벡터로 바꾼다.
        #
        # (2,) -> (2, 1)
        weight_column = weights.reshape(-1, 1)

        # 잡음이 없는 정확한 선형관계를 계산한다.
        #
        # (2000, 2) @ (2, 1) + (1)
        # -> (2000, 1)
        self.clean_y = self.X @ weight_column + bias

        # 각 example에 독립적인 Gaussian noise를 생성한다.
        #
        # epsilon ~ N(0, noise_std^2)
        self.noise = (
            torch.randn(number_of_examples, 1)
            * noise_std
        )
        
        # 실제 관측 label은 정확한 선형관계에 잡음을 더한 값이다.
        self.y = self.clean_y + self.noise

In [ ]:
# Cell 3
# 우리가 알고 있는 실제 parameter
true_weights = torch.tensor([2.0, -3.4])
true_bias = 4.2

data = SyntheticRegressionData(
    weights=true_weights,
    bias=true_bias,
    noise_std=0.01,
    num_train=1000,
    num_val=1000,
    batch_size=32,
)

print("X shape:", data.X.shape)
print("y shape:", data.y.shape)

assert data.X.shape == (2000, 2)
assert data.y.shape == (2000, 1)

In [ ]:
# Cell 4
# 첫 번째 sample의 feature, 잡음 없는 label,
# 실제 noise와 최종 관측 label을 확인한다.
first_features = data.X[0]
first_clean_label = data.clean_y[0]
first_noise = data.noise[0]
first_observed_label = data.y[0]

print("First features:", first_features)
print("First clean label:", first_clean_label)
print("First noise:", first_noise)
print("First observed label:", first_observed_label)

# observed label = clean label + noise
assert torch.allclose(
    first_observed_label,
    first_clean_label + first_noise,
)

In [ ]:
# Cell 5
# 생성한 noise가 평균 약 0, 표준편차 약 0.01인지 확인한다.
empirical_noise_mean = data.noise.mean()
empirical_noise_std = data.noise.std()

print("Noise mean:", empirical_noise_mean.item())
print("Noise standard deviation:", empirical_noise_std.item())